In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,operator,Annotated
from pydantic import BaseModel,Field
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import ToolNode

import redis
import json
import time

In [13]:
async def run_chatbot():
    import sys
    from pathlib import Path
    from langchain_mcp_adapters.client import MultiServerMCPClient
    
    MCP_SERVER_PATH = "/Users/nehadubey/Desktop/langgraph_tutorial/mcp_server.py"
    
    mcp_config = {
        "notes_and_tools": {
            "command": sys.executable,
            "args": [MCP_SERVER_PATH],
            "transport": "stdio",
        }
    }
    
    print(sys.executable)
    print(MCP_SERVER_PATH)
    
    client = MultiServerMCPClient(mcp_config)
    tools = await client.get_tools()
    
    print([t.name for t in tools])
    model=ChatOllama(model='llama3.2:latest')
    model_with_tools = model.bind_tools(tools)
    r = redis.Redis(host='localhost', port=6379, decode_responses=True)

    class ChatState(TypedDict):
        messages : Annotated[list[BaseMessage],add_messages]

    def chat_node(state: ChatState):
        messages = state['messages']
        start_time = time.time()
        response = model_with_tools.invoke(messages)
        total_client_ms = (time.time() - start_time) * 1000
    
        # Agar tool call hai toh metadata nahi hota — sirf client time dikhao
        if response.tool_calls:
            print(f"  🔧 Tool call: {response.tool_calls[0]['name']}({response.tool_calls[0]['args']})")
            print(f"  ⏱ Client time: {total_client_ms:.1f}ms")
            return {'messages': [response]}
    
        # Normal response — server metadata available hai
        meta = response.response_metadata
        load_ms   = meta['load_duration'] / 1e6
        prompt_ms = meta['prompt_eval_duration'] / 1e6
        gen_ms    = meta['eval_duration'] / 1e6
        total_ms  = meta['total_duration'] / 1e6
        ttft_ms   = load_ms + prompt_ms
    
        tokens = meta['eval_count']
        speed = tokens / (meta['eval_duration'] / 1e9) if meta['eval_duration'] > 0 else 0
    
        # Redis mein store karo
        # Redis mein store karo - sab values str() mein wrap karo
        r.xadd("ttft:chatbot", {
            "ttft_ms": str(round(ttft_ms, 2)),
            "load_ms": str(round(load_ms, 2)),
            "prompt_ms": str(round(prompt_ms, 2)),
            "gen_ms": str(round(gen_ms, 2)),
            "total_ms": str(round(total_ms, 2)),
            "client_total_ms": str(round(total_client_ms, 2)),
            "tokens": str(tokens),
            "speed_tps": str(round(speed, 2)),
            "prompt_tokens": str(meta.get('prompt_eval_count', 0)),
            "user_query": str(messages[-1].content[:100]),
        })    
        print(f"  ⏱ TTFT: {ttft_ms:.1f}ms | Gen: {gen_ms:.1f}ms | Speed: {speed:.1f} tok/s")
        return {'messages': [response]}

    # Tool node - actual tool execute karta hai via MCP
    tool_node = ToolNode(tools)
    # Router - decide karo tool call karna hai ya seedha reply
    def should_use_tool(state: ChatState) -> Literal["tools", END]:
        last_message = state['messages'][-1]
        if last_message.tool_calls:
            return "tools"  # tool use karo
        return END          # seedha reply de do
    # Router - decide karo tool call karna hai ya seedha reply
    def should_use_tool(state: ChatState) -> Literal["tools", END]:
        last_message = state['messages'][-1]
        if last_message.tool_calls:
            return "tools"  # tool use karo
        return END          # seedha reply de do
 
        
    # --- Step E: Graph banao ---
    graph = StateGraph(ChatState)
    
    # Nodes add karo
    graph.add_node("chatbot", chat_node)
    graph.add_node("tools", tool_node)
    
    # Edges add karo
    graph.add_edge(START, "chatbot")
    graph.add_conditional_edges("chatbot", should_use_tool)
    graph.add_edge("tools", "chatbot")  # tool ke baad wapas chatbot pe aao
    
    # Compile
    checkpointer = MemorySaver()
    chatbot = graph.compile(checkpointer=checkpointer)
    
    thread_id = "mcp-session-1"
    
    while True:
        user_message=input("type_here:" )
        print(user_message)
        if user_message.strip().lower() in ['exit','quit','bye']:
            break
        config={'configurable':{'thread_id':thread_id}}
        response=await chatbot.ainvoke({'messages':[HumanMessage(content=user_message)]},config=config)
        # Saare messages dekho - tool result bhi dikhega
        for msg in response['messages']:
            print(f"  [{msg.__class__.__name__}]: {msg.content[:200]}")

        print('AI:',response['messages'][-1].content)

In [14]:
await run_chatbot()

/Users/nehadubey/Desktop/langgraph_tutorial/myenv/bin/python
/Users/nehadubey/Desktop/langgraph_tutorial/mcp_server.py
['save_note', 'get_notes', 'delete_note', 'calculator', 'get_datetime']


type_here: current time


current time
  🔧 Tool call: get_datetime({})
  ⏱ Client time: 1012.5ms
  ⏱ TTFT: 216.9ms | Gen: 222.5ms | Speed: 49.4 tok/s
  [HumanMessage]: current time
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:02, Day: Wednesday', 'id': 'lc_64d088a6-338b-4ac7-bc8e-b0b4ecf8d99f'}]
  [AIMessage]: The current time is 2:36 PM.
AI: The current time is 2:36 PM.


type_here: current date


current date
  🔧 Tool call: get_datetime({})
  ⏱ Client time: 999.7ms
  ⏱ TTFT: 214.6ms | Gen: 268.2ms | Speed: 48.5 tok/s
  [HumanMessage]: current time
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:02, Day: Wednesday', 'id': 'lc_64d088a6-338b-4ac7-bc8e-b0b4ecf8d99f'}]
  [AIMessage]: The current time is 2:36 PM.
  [HumanMessage]: current date
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:19, Day: Wednesday', 'id': 'lc_3881b52c-dd78-4392-ab94-8cb55e1a35c1'}]
  [AIMessage]: The current date is June 17, 2026.
AI: The current date is June 17, 2026.


...

You: save a note titled shopping with content buy milk and bread You: save a note titled work with content finish presentation by friday You: show me all my notes
  🔧 Tool call: save_note({'title': 'shopping', 'content': 'buy milk and bread'})
  ⏱ Client time: 2349.3ms
  ⏱ TTFT: 469.3ms | Gen: 703.4ms | Speed: 44.1 tok/s
  [HumanMessage]: current time
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:02, Day: Wednesday', 'id': 'lc_64d088a6-338b-4ac7-bc8e-b0b4ecf8d99f'}]
  [AIMessage]: The current time is 2:36 PM.
  [HumanMessage]: current date
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:19, Day: Wednesday', 'id': 'lc_3881b52c-dd78-4392-ab94-8cb55e1a35c1'}]
  [AIMessage]: The current date is June 17, 2026.
  [HumanMessage]: ok save this time in a file
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': "Error executing tool save_note: 1 validation error for save_noteArguments\nco

type_here: show me all my notes


show me all my notes
  🔧 Tool call: get_notes({})
  ⏱ Client time: 1042.4ms
  ⏱ TTFT: 224.3ms | Gen: 513.3ms | Speed: 44.8 tok/s
  [HumanMessage]: current time
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:02, Day: Wednesday', 'id': 'lc_64d088a6-338b-4ac7-bc8e-b0b4ecf8d99f'}]
  [AIMessage]: The current time is 2:36 PM.
  [HumanMessage]: current date
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': 'Current date: 2026-06-17, Time: 14:36:19, Day: Wednesday', 'id': 'lc_3881b52c-dd78-4392-ab94-8cb55e1a35c1'}]
  [AIMessage]: The current date is June 17, 2026.
  [HumanMessage]: ok save this time in a file
  [AIMessage]: 
  [ToolMessage]: [{'type': 'text', 'text': "Error executing tool save_note: 1 validation error for save_noteArguments\ncontent\n  Input should be a valid string [type=string_type, input_value={'time': 'Current time sa...at 2026-06-17 14:38:33'}, input_type=dict]\n    For further information visit https://errors.py

type_here: exit


exit
